<a href="https://colab.research.google.com/github/Pranathitejasvi/Automatic-Machine-translation-of-Medical-transcription/blob/main/NLP_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentencepiece torch googletrans==4.0.0-rc1 bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
%%writefile medical_translator.py
import re
import time
import logging
from typing import Optional
from dataclasses import dataclass, field

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

@dataclass
class TranslationResult:
    original_text: str
    translated_text: str
    raw_translated_text: Optional[str] = None # Added this line
    source_language: str = "en"
    target_language: str = "hi"
    model_used: str = ""
    processing_time_seconds: float = 0.0
    detected_medical_terms: list = field(default_factory=list)
    confidence_score: float = 0.0
    segments: list = field(default_factory=list)

MEDICAL_GLOSSARY = {
    "hi": {
        "hypertension": "उच्च रक्तचाप", "diabetes": " मधुमेह", "fever": "बुखार",
        "headache": "सिरदर्द", "chest pain": "सीने में दर्द", "blood pressure": "रक्तचाप",
        "heart rate": "हृदय गति", "prescription": "नुस्खा", "diagnosis": "निदान",
        "symptoms": "लक्षण", "treatment": "उपचार", "medication": "दवा",
        "dosage": "खुराक", "allergy": "एलर्जी", "surgery": "शल्य चिकित्सा",
        "fracture": "हड्डी टूटना", "infection": "संक्रमण", "inflammation": "सूजन",
        "patient": "रोगी", "doctor": "चिकित्सक", "hospital": "अस्पताल",
        "emergency": "आपातकाल", "ambulance": "एम्बुलेंस", "x-ray": "एक्स-रे",
        "ultrasound": "अल्ट्रासाउंड", "blood test": "रक्त परीक्षण", "urine test": "मूत्र परीक्षण",
        "cholesterol": "कोलेस्ट्रॉल", "anemia": "रक्ताल्पता", "asthma": "दमा",
        "pneumonia": "निमोनिया", "tuberculosis": "तपेदिक", "cancer": "कैंसर",
        "stroke": "आघात", "kidney": "गुर्दा", "liver": "यकृत", "lung": "फेफड़ा",
        "heart": "हृदय", "brain": "मस्तिष्क", "tablet": "गोली", "capsule": "कैप्सूल",
        "injection": "इंजेक्शन", "antibiotic": "एंटीबायोटिक", "painkiller": "दर्द निवारक",
        "insulin": "इंसुलिन",
        "pain": "दर्द", "disease": "रोग", "virus": "वायरस", "bacteria": "बैक्टीरिया",
        "vaccine": "टीका", "therapy": "चिकित्सा", "chronic": "पुरानी",
        "acute": "तीव्र", "cough": "खांसी", "cold": "सर्दी", "swelling": "सूजन",
        "bleeding": "रक्तस्राव", "wound": "घाव", "nausea": "मतली",
        "vomiting": "उल्टी", "diarrhea": "दस्त", "constipation": "कब्ज"
    }
}

SUPPORTED_LANGUAGES = {
    "hi": {"name": "Hindi", "model": "Helsinki-NLP/opus-mt-en-hi"},
}

class MedicalTranscriptionTranslator:
    def __init__(self, target_language: str = "hi"):
        if target_language not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language '{target_language}'. Choose: {list(SUPPORTED_LANGUAGES)}")
        self.target_language = target_language
        self.lang_info = SUPPORTED_LANGUAGES[target_language]
        self.glossary = MEDICAL_GLOSSARY.get(target_language, {})
        self.model = None
        self.tokenizer = None
        self._load_model()

    def _load_model(self):
        try:
            from transformers import MarianMTModel, MarianTokenizer
            import torch
            model_name = self.lang_info["model"]
            logger.info(f"Loading model: {model_name} ...")
            self.tokenizer = MarianTokenizer.from_pretrained(model_name)
            self.model     = MarianMTModel.from_pretrained(model_name)
            if torch.cuda.is_available():
                self.model = self.model.to("cuda")
            self._use_google_fallback = False
            logger.info("Model loaded successfully.")
        except Exception as e:
            logger.warning(f"Model load failed ({e}). Using Google Translate fallback.")
            self._use_google_fallback = True

    def _preprocess(self, text: str) -> str:
        text = re.sub(r"\s+", " ", text).strip()
        # Expanded shortcuts to safeguard translation flow
        abbreviations = {
            r"\bpt\b": "patient",      r"\bdx\b": "diagnosis",
            r"\brx\b": "prescription", r"\bhx\b": "history",
            r"\bbp\b": "blood pressure", r"\bhr\b": "heart rate",
            r"\btemp\b": "temperature",  r"\bwt\b": "weight",
            r"\bht\b": "height",         r"\bbid\b": "twice daily",
            r"\btid\b": "three times daily", r"\bqid\b": "four times daily",
            r"\bprn\b": "as needed",     r"\bstat\b": "immediately",
            r"\bpo\b": "by mouth",       r"\biv\b": "intravenous",
            r"\bim\b": "intramuscular",
            r"\brice\b": "Rest, Ice, Compression, Elevation",
            r"\bmcl\b": "medial collateral ligament sprain"
        }
        for pattern, replacement in abbreviations.items():
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text

    def _detect_medical_terms(self, text: str) -> list:
        text_lower = text.lower()
        return [term for term in self.glossary if term in text_lower]

    def _split_into_segments(self, text: str, max_chars: int = 400) -> list:
        sentences = re.split(r"(?<=[.!?])\s+", text)
        segments, current = [], ""
        for sentence in sentences:
            if len(current) + len(sentence) + 1 <= max_chars:
                current = (current + " " + sentence).strip()
            else:
                if current:
                    segments.append(current)
                current = sentence
        if current:
            segments.append(current)
        return segments or [text]

    def _apply_glossary(self, translated: str, original: str) -> str:
        original_lower = original.lower()
        # Clean up any potential token loops or repetitive characters generated by the model
        translated = re.sub(r'([a-zA-Z0-9])\1{4,}', r'\1', translated)
        for eng_term in sorted(self.glossary, key=len, reverse=True):
            if eng_term in original_lower:
                regional_term = self.glossary[eng_term]
                if regional_term not in translated:
                    # Injects terms smoothly as standard localized parentheses instead of messy trailing arrays
                    translated = f"{translated} ({regional_term})"
        return translated

    def _translate_with_model(self, text: str) -> str:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            ids = self.model.generate(**inputs, num_beams=4, max_length=512, early_stopping=True)
        return self.tokenizer.decode(ids[0], skip_special_tokens=True)

    def _translate_with_google(self, text: str) -> str:
        from googletrans import Translator
        return Translator().translate(text, src="en", dest=self.target_language).text

    def translate(self, text: str) -> TranslationResult:
        start = time.time()
        clean  = self._preprocess(text)
        terms  = self._detect_medical_terms(clean)
        segs   = self._split_into_segments(clean)
        translated_segs = []
        raw_translated_segs = [] # New list for raw translations
        for seg in segs:
            t_raw = self._translate_with_google(seg) if self._use_google_fallback else self._translate_with_model(seg)
            raw_translated_segs.append(t_raw) # Store raw translation
            t_glossary = self._apply_glossary(t_raw, seg)
            translated_segs.append(t_glossary)

        return TranslationResult(
            original_text=text,
            translated_text=" ".join(translated_segs),
            raw_translated_text=" ".join(raw_translated_segs), # Populate new field
            source_language="en",
            target_language=self.target_language,
            model_used="google-translate" if self._use_google_fallback else self.lang_info["model"],
            processing_time_seconds=round(time.time() - start, 2),
            detected_medical_terms=terms,
            confidence_score=round(max(0.0, 1.0 - 0.05 * len(terms)), 2),
            segments=list(zip(segs, translated_segs)),
        )

    def translate_batch(self, texts: list) -> list:
        return [self.translate(t) for t in texts]

    def translate_report(self, report: dict) -> dict:
        translated_report = {}
        for section, original_text in report.items():
            translation_result = self.translate(original_text)
            translated_report[section] = {
                "original": original_text,
                "translated": translation_result.translated_text,
                "raw_translated": translation_result.raw_translated_text,
                "detected_medical_terms": translation_result.detected_medical_terms
            }
        return translated_report

In [ ]:
%%writefile evaluation.py
import re
import math
import json # Import json for history persistence
from collections import Counter
from dataclasses import dataclass
from typing import Optional, List, Dict
from medical_translator import MedicalTranscriptionTranslator, MEDICAL_GLOSSARY
from bert_score import score # Import bert_score

# --- Helper functions for history persistence ---
def save_evaluation_history(history: List[Dict], filename: str = "translation_history.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

def load_evaluation_history(filename: str = "translation_history.json") -> List[Dict]:
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def calculate_bert_score(hypothesis: List[str], reference: List[str]) -> List[Dict]:
    # Using a multilingual model that works well for English-Hindi
    # 'microsoft/mdeberta-v3-base' is a good general-purpose multilingual model.
    # We typically report F1 score for BERTScore.
    # The 'score' function returns P, R, F1 tensors. F1 is a tensor of scores for each sentence pair.
    P, R, F1 = score(hypothesis, reference, lang='en', model_type="microsoft/mdeberta-v3-base", verbose=True)
    return [{'p': round(p.item(), 4), 'r': round(r.item(), 4), 'f1': round(f1.item(), 4)} for p, r, f1 in zip(P, R, F1)] # Return a list of dicts with P, R, F1 scores

def medical_term_accuracy(original: str, translation: str, lang: str) -> dict:
    glossary = MEDICAL_GLOSSARY.get(lang, {})
    found_terms_in_original = [term for term in glossary if term in original.lower()]
    correctly_translated_terms = [term for term in found_terms_in_original if glossary[term] in translation]
    missed_terms = [term for term in found_terms_in_original if term not in correctly_translated_terms]

    return {
        "total_medical_terms_in_original": len(found_terms_in_original),
        "correctly_translated_terms_count": len(correctly_translated_terms),
        "accuracy": round(len(correctly_translated_terms)/len(found_terms_in_original), 4) if found_terms_in_original else 1.0,
        "missed_terms": missed_terms,
        "correct_terms": correctly_translated_terms,
    }

@dataclass
class EvalResult:
    sample_id: int
    source_text: str
    hypothesis: str
    reference: Optional[str]
    bert_score_p: Optional[float] # Added BERTScore P
    bert_score_r: Optional[float] # Added BERTScore R
    bert_score_f1: Optional[float] # Added BERTScore F1
    medical_accuracy: dict
    processing_time: float
    target_lang: str

    def to_dict(self):
        return {
            "sample_id": self.sample_id,
            "source_text": self.source_text,
            "hypothesis": self.hypothesis,
            "reference": self.reference,
            "bert_score_p": self.bert_score_p, # Include in dict
            "bert_score_r": self.bert_score_r, # Include in dict
            "bert_score_f1": self.bert_score_f1, # Include in dict
            "medical_accuracy": self.medical_accuracy,
            "processing_time": self.processing_time,
            "target_lang": self.target_lang,
        }

    @classmethod
    def from_dict(cls, data: Dict):
        return cls(
            sample_id=data["sample_id"],
            source_text=data["source_text"],
            hypothesis=data["hypothesis"],
            reference=data["reference"],
            bert_score_p=data.get("bert_score_p"), # Get with .get() for backward compatibility
            bert_score_r=data.get("bert_score_r"), # Get with .get() for backward compatibility
            bert_score_f1=data.get("bert_score_f1"), # Get with .get() for backward compatibility
            medical_accuracy=data["medical_accuracy"],
            processing_time=data["processing_time"],
            target_lang=data["target_lang"],
        )

class TranslationEvaluator:
    def __init__(self, target_language: str = "hi"):
        self.translator      = MedicalTranscriptionTranslator(target_language)
        self.target_language = target_language

    def evaluate_dataset(self, dataset: list) -> List[EvalResult]:
        results = []
        hypotheses = []
        references = []
        source_texts = []

        for item in dataset:
            src = item["source"]
            ref = item.get("reference")
            res = self.translator.translate(src)
            hypotheses.append(res.translated_text)
            references.append(ref)
            source_texts.append(src)

        bert_scores = []
        if all(r is not None for r in references):
            # Calculate BERTScore once for all hypotheses and references
            bert_scores = calculate_bert_score(hypotheses, references)

        for idx, (src, ref, hypothesis) in enumerate(zip(source_texts, references, hypotheses)):
            res = self.translator.translate(src) # Re-translate to get other metrics
            current_bert_score = bert_scores[idx] if bert_scores else {'p': None, 'r': None, 'f1': None}
            results.append(EvalResult(
                sample_id=idx,
                source_text=src,
                hypothesis=hypothesis,
                reference=ref,
                bert_score_p=current_bert_score['p'], # Assign BERTScore P
                bert_score_r=current_bert_score['r'], # Assign BERTScore R
                bert_score_f1=current_bert_score['f1'], # Assign BERTScore F1
                medical_accuracy=medical_term_accuracy(src, hypothesis, self.target_language),
                processing_time=res.processing_time_seconds,
                target_lang=self.target_language,
            ))
        return results

    def aggregate_metrics(self, results: List[EvalResult]) -> dict:
        bert_scores_p = [r.bert_score_p for r in results if r.bert_score_p is not None]
        bert_scores_r = [r.bert_score_r for r in results if r.bert_score_r is not None]
        bert_scores_f1 = [r.bert_score_f1 for r in results if r.bert_score_f1 is not None]
        return {
            "num_samples":                  len(results),
            "avg_bert_score_p":             round(sum(bert_scores_p)/len(bert_scores_p), 4) if bert_scores_p else "N/A", # Add average BERTScore P
            "avg_bert_score_r":             round(sum(bert_scores_r)/len(bert_scores_r), 4) if bert_scores_r else "N/A", # Add average BERTScore R
            "avg_bert_score_f1":            round(sum(bert_scores_f1)/len(bert_scores_f1), 4) if bert_scores_f1 else "N/A", # Add average BERTScore F1
            "avg_medical_term_accuracy":    round(sum(r.medical_accuracy["accuracy"] for r in results)/len(results), 4) if results else "N/A",
            "avg_processing_time_seconds":  round(sum(r.processing_time for r in results)/len(results), 3) if results else "N/A",
        }

    def generate_html_report(self, results: List[EvalResult], output_path: str = "report.html"):
        if not results:
            print("No evaluation results to generate report.")
            return

        m = self.aggregate_metrics(results)
        rows = "".join(f"""
          <tr>
            <td>{r.sample_id}</td>
            <td>{r.source_text[:100]}…</td>
            <td>{r.hypothesis[:100]}…</td>
            <td>{f"{r.bert_score_p:.4f}" if r.bert_score_p is not None else "—"}</td> <!-- Add BERTScore P to table -->
            <td>{f"{r.bert_score_r:.4f}" if r.bert_score_r is not None else "—"}</td> <!-- Add BERTScore R to table -->
            <td>{f"{r.bert_score_f1:.4f}" if r.bert_score_f1 is not None else "—"}</td> <!-- Add BERTScore F1 to table -->
            <td>{r.medical_accuracy['accuracy']:.2%}</td>
            <td>{r.processing_time:.2f}s</td>
          </tr>""" for r in results)

        html = f"""<!DOCTYPE html><html><head><meta charset=\"UTF-8\"><title>Medical Translation Report</title><style>  body{{font-family:sans-serif;background:#f8fafc;padding:2rem;color:#1e293b}}  h1{{border-bottom:3px solid #3b82f6;padding-bottom:.5rem}}  .metrics{{display:flex;gap:1rem;flex-wrap:wrap;margin:1.5rem 0}}  .card{{background:white;border-radius:10px;padding:1rem 2rem;box-shadow:0 2px 8px rgba(0,0,0,.07);text-align:center}}  .value{{font-size:2rem;font-weight:700;color:#3b82f6}}  .label{{font-size:.85rem;color:#64748b}}  table{{width:100%;border-collapse:collapse;background:white;border-radius:10px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.07)}}  th{{background:#1e40af;color:white;padding:.7rem 1rem;text-align:left}}  td{{padding:.6rem 1rem;border-bottom:1px solid #e2e8f0;font-size:.83rem}}  tr:hover td{{background:#eff6ff}}</style></head><body><h1>🏥 Medical Translation — Evaluation Report</h1><div class="metrics">  <div class="card"><div class="value">{m['num_samples']}</div><div class="label">Samples</div></div>  <div class="card"><div class="value">{m['avg_bert_score_p']}</div><div class="label">Avg BERTScore P</div></div>  <div class="card"><div class="value">{m['avg_bert_score_r']}</div><div class="label">Avg BERTScore R</div></div>  <div class="card"><div class="value">{m['avg_bert_score_f1']}</div><div class="label">Avg BERTScore F1</div></div>  <div class="card"><div class="value">{m['avg_medical_term_accuracy']:.2%}</div><div class="label">Med. Term Accuracy</div></div>  <div class="card"><div class="value">{m['avg_processing_time_seconds']}s</div><div class="label">Avg Time</div></div></div><table><thead><tr><th>#</th><th>Source</th><th>Translation</th><th>BERTScore P</th><th>BERTScore R</th><th>BERTScore F1</th><th>Med. Acc.</th><th>Time</th></tr></thead><tbody>{rows}</tbody></table></body></html>"""

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"Report saved → {output_path}")

In [ ]:
%%writefile demo.py
import argparse
import os # Import os for file path checks
from medical_translator import MedicalTranscriptionTranslator
from evaluation import medical_term_accuracy, TranslationEvaluator, EvalResult, save_evaluation_history, load_evaluation_history, calculate_bert_score # Import calculate_bert_score
import re # for gibberish check

HISTORY_FILE = "translation_history.json"

SAMPLES = [
    """Patient is a 45-year-old male with chest pain and shortness of breath.
    History of hypertension and diabetes. BP 150/95. HR 88.
    Prescription: Metformin 500 mg bid, Amlodipine 5 mg once daily.""",
    """32-year-old female with fever and headache for 5 days.
    Diagnosis: Viral fever. Treatment: Paracetamol 500 mg prn,
    adequate hydration. Return if rash or bleeding develops.""",
    """Knee pain after sports injury. X-ray: no fracture.
    Impression: Grade II MCL sprain. Treatment: RICE,
    Ibuprofen 400 mg tid, physiotherapy referral.""",
]

REPORT = {
    "chief_complaint": "Severe headache with nausea and vomiting for 2 days",
    "history":         "History of migraine and hypertension. No known drug allergy.",
    "diagnosis":       "Hypertensive urgency with migraine",
    "treatment":       "Amlodipine 10 mg once daily. Sumatriptan 50 mg prn for migraine.",
    "follow_up":       "Review in one week. Monitor blood pressure daily.",
}

EVAL_DATA = [
    {"source": "Patient has fever and headache. Diagnosis: viral infection.",
     "reference": "रोगी को बुखार और सिरदर्द है। निदान: वायरल संक्रमण।"},
    {"source": "Blood pressure is high. Prescription: antihypertensive tablet once daily.",
     "reference": "रक्तचाप अधिक है। नुस्खा: उच्चरक्तचापरोधी गोली दिन में एक बार।"},
    {"source": "Patient has diabetes. Take insulin injection as prescribed.",
     "reference": "रोगी को मधुमेह है। निर्देशानुसार इंसुलिन इंजेक्शन लें।"},
]

def contains_hindi(text):
    # Basic check if the text contains Hindi characters
    return bool(re.search(r'[क-ह]', text))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--eval", action="store_true")
    parser.add_argument("--custom_text", type=str, help="Custom English text to translate.")
    parser.add_argument("--reference_text", type=str, help="Optional reference Hindi text for custom translation evaluation.")
    args = parser.parse_args()

    translator = MedicalTranscriptionTranslator(target_language="hi")
    lang_name  = "Hindi"

    if args.custom_text:
        print(f"\n{'═'*65}")
        print(f"  Interactive Medical Translation  |  English → {lang_name}")
        print(f"{'═'*65}")
        print("\nNote on input quality:")
        print("  - For best results, keep the input text concise (e.g., 1-2 sentences).")
        print("  - Focus on medical terminology relevant to the glossary (e.g., 'hypertension', 'fracture').")
        print("  - Avoid overly complex sentence structures or general conversation.")
        print("  - Extremely long texts or non-medical content might lead to less accurate translations or irrelevant glossary applications.")

        print(f"\n[Custom Input]")
        custom_input = args.custom_text
        print(f"  EN : {custom_input}")

        # Perform translation
        translation_result = translator.translate(custom_input)

        # Gibberish check
        # A simple heuristic: if raw translated text is very short OR
        # if it doesn't contain any Hindi characters and is significantly different from original (or too short),
        # it might indicate a translation failure.
        if (len(translation_result.raw_translated_text.strip()) < len(custom_input.strip()) * 0.2 and len(custom_input.strip()) > 10) or \
           (not contains_hindi(translation_result.raw_translated_text) and custom_input.strip().lower() != translation_result.raw_translated_text.strip().lower()):
            print("\n⚠️  WARNING: The translated text appears to be gibberish or very poor quality. Please check your input and consider the guidelines.")
            # We will still proceed to record it for history, but warn the user.

        print(f"  HI (Model w/o Glossary): {translation_result.raw_translated_text}")
        print(f"  HI (Model w/ Glossary) : {translation_result.translated_text}")
        print(f"  Terms detected         : {translation_result.detected_medical_terms}")
        print(f"  Time                   : {translation_result.processing_time_seconds}s")

        medical_acc = medical_term_accuracy(custom_input, translation_result.translated_text, translator.target_language)
        print(f"  Medical Term Accuracy: {medical_acc['accuracy']:.2%} (Missed: {medical_acc['missed_terms']})")

        current_bert_score_p = None
        current_bert_score_r = None
        current_bert_score_f1 = None # Initialize BERTScore
        if args.reference_text:
            # Calculate BERTScore for custom translation
            bert_score_results = calculate_bert_score([translation_result.translated_text], [args.reference_text])
            if bert_score_results:
                current_bert_score_p = bert_score_results[0]['p']
                current_bert_score_r = bert_score_results[0]['r']
                current_bert_score_f1 = bert_score_results[0]['f1'] # Extract the single F1 score
            print(f"  BERTScore P            : {current_bert_score_p:.4f}")
            print(f"  BERTScore R            : {current_bert_score_r:.4f}")
            print(f"  BERTScore F1           : {current_bert_score_f1:.4f}")
            print(f"  Reference Text         : {args.reference_text}")

        # Load existing history
        history_dicts = load_evaluation_history(HISTORY_FILE)
        current_id = len(history_dicts) # Assign new ID for current entry

        # Create an EvalResult for the current translation and convert to dict for saving
        current_eval_result = EvalResult(
            sample_id=current_id,
            source_text=custom_input,
            hypothesis=translation_result.translated_text,
            reference=args.reference_text, # Use provided reference
            bert_score_p=current_bert_score_p, # Store BERTScore P
            bert_score_r=current_bert_score_r, # Store BERTScore R
            bert_score_f1=current_bert_score_f1, # Store BERTScore F1
            medical_accuracy=medical_acc,
            processing_time=translation_result.processing_time_seconds,
            target_lang=translator.target_language,
        )
        history_dicts.append(current_eval_result.to_dict())
        save_evaluation_history(history_dicts, HISTORY_FILE)
        print(f"Translation saved to history: {HISTORY_FILE}")

        # Regenerate report from full history
        all_eval_results = [EvalResult.from_dict(d) for d in history_dicts]
        evaluator = TranslationEvaluator(target_language=translator.target_language) # Re-initialize to ensure it has the translator
        evaluator.generate_html_report(all_eval_results, "report_hi.html")
        print(f"Updated HTML report → report_hi.html with all custom translations.")


        print("\n✅  Done with custom translation.\n")

    else: # Existing logic for SAMPLES, REPORT, batch, eval
        print(f"\n{'═'*65}")
        print(f"  Medical Transcription Translator  |  English → {lang_name}")
        print(f"{'═'*65}")
        for i, text in enumerate(SAMPLES):
            result = translator.translate(text)
            print(f"\n[Sample {i+1}]")
            print(f"  EN : {result.original_text.strip()[:180]}\n") # Added newline for clarity
            print(f"  HI : {result.translated_text}\n") # Added newline for clarity
            print(f"  Terms detected : {result.detected_medical_terms}")
            print(f"  Time : {result.processing_time_seconds}s")

        print(f"\n{'─'*65}\n  Structured Medical Report\n{'─'*65}")
        report_out = translator.translate_report(REPORT)
        for section, data in report_out.items():
            print(f"\n  {section.upper()}")
            print(f"    EN : {data['original']}\n") # Added newline for clarity
            print(f"    HI : {data['translated']}\n") # Added newline for clarity

        print(f"\n{'─'*65}\n  Batch Translation\n{'─'*65}")
        batch = [
            "Take this antibiotic capsule twice daily for seven days.",
            "Patient was discharged after successful surgery.",
            "Allergy to penicillin noted. Dosage of medication adjusted.",
        ]
        for r in translator.translate_batch(batch):
            print(f"\n  EN : {r.original_text}\n") # Added newline for clarity
            print(f"  HI : {r.translated_text}\n") # Added newline for clarity

        if args.eval:
            print(f"\n{'─'*65}\n  Evaluation Metrics\n{'─'*65}")
            evaluator = TranslationEvaluator(target_language="hi")
            results   = evaluator.evaluate_dataset(EVAL_DATA)
            metrics   = evaluator.aggregate_metrics(results)

            print(f"\n  Avg BERTScore P         : {metrics['avg_bert_score_p']}") # Print average BERTScore P
            print(f"  Avg BERTScore R         : {metrics['avg_bert_score_r']}") # Print average BERTScore R
            print(f"  Avg BERTScore F1        : {metrics['avg_bert_score_f1']}") # Print average BERTScore F1
            print(f"  Avg Medical Term Acc.   : {metrics['avg_medical_term_accuracy']:.2%}")
            print(f"  Avg Processing Time     : {metrics['avg_processing_time_seconds']}s")

            for r in results:
                print(f"\n  [{r.sample_id}] BERTScore P={r.bert_score_p} | BERTScore R={r.bert_score_r} | BERTScore F1={r.bert_score_f1} | Med.Acc={r.medical_accuracy['accuracy']:.2%}") # Print individual BERTScore
                if r.medical_accuracy["missed_terms"]:
                    print(f"      ⚠ Missed: {r.medical_accuracy['missed_terms']}")

            evaluator.generate_html_report(results, "report_hi.html")
            print(f"\n  HTML report → report_hi.html")

        print("\n\n✅  Done.\n")

if __name__ == "__main__":
    main()

In [ ]:
import bert_score
print('bert_score is importable!')
!python demo.py --custom_text "Patient presents with acute abdominal pain and nausea. CT scan shows appendicitis." --reference_text "रोगी तीव्र पेट दर्द और मतली के साथ प्रस्तुत होता है। सीटी स्कैन में एपेंडिसाइटिस दिखाया गया है।"

In [ ]:
%%writefile medical_translator.py
import re
import time
import logging
from typing import Optional
from dataclasses import dataclass, field

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

@dataclass
class TranslationResult:
    original_text: str
    translated_text: str
    raw_translated_text: Optional[str] = None # Added this line
    source_language: str = "en"
    target_language: str = "hi"
    model_used: str = ""
    processing_time_seconds: float = 0.0
    detected_medical_terms: list = field(default_factory=list)
    confidence_score: float = 0.0
    segments: list = field(default_factory=list)

MEDICAL_GLOSSARY = {
    "hi": {
        "hypertension": "उच्च रक्तचाप", "diabetes": " मधुमेह", "fever": "बुखार",
        "headache": "सिरदर्द", "chest pain": "सीने में दर्द", "blood pressure": "रक्तचाप",
        "heart rate": "हृदय गति", "prescription": "नुस्खा", "diagnosis": "निदान",
        "symptoms": "लक्षण", "treatment": "उपचार", "medication": "दवा",
        "dosage": "खुराक", "allergy": "एलर्जी", "surgery": "शल्य चिकित्सा",
        "fracture": "हड्डी टूटना", "infection": "संक्रमण", "inflammation": "सूजन",
        "patient": "रोगी", "doctor": "चिकित्सक", "hospital": "अस्पताल",
        "emergency": "आपातकाल", "ambulance": "एम्बुलेंस", "x-ray": "एक्स-रे",
        "ultrasound": "अल्ट्रासाउंड", "blood test": "रक्त परीक्षण", "urine test": "मूत्र परीक्षण",
        "cholesterol": "कोलेस्ट्रॉल", "anemia": "रक्ताल्पता", "asthma": "दमा",
        "pneumonia": "निमोनिया", "tuberculosis": "तपेदिक", "cancer": "कैंसर",
        "stroke": "आघात", "kidney": "गुर्दा", "liver": "यकृत", "lung": "फेफड़ा",
        "heart": "हृदय", "brain": "मस्तिष्क", "tablet": "गोली", "capsule": "कैप्सूल",
        "injection": "इंजेक्शन", "antibiotic": "एंटीबायोटिक", "painkiller": "दर्द निवारक",
        "insulin": "इंसुलिन",
        "pain": "दर्द", "disease": "रोग", "virus": "वायरस", "bacteria": "बैक्टीरिया",
        "vaccine": "टीका", "therapy": "चिकित्सा", "chronic": "पुरानी",
        "acute": "तीव्र", "cough": "खांसी", "cold": "सर्दी", "swelling": "सूजन",
        "bleeding": "रक्तस्राव", "wound": "घाव", "nausea": "मतली",
        "vomiting": "उल्टी", "diarrhea": "दस्त", "constipation": "कब्ज"
    }
}

SUPPORTED_LANGUAGES = {
    "hi": {"name": "Hindi", "model": "Helsinki-NLP/opus-mt-en-hi"},
}

class MedicalTranscriptionTranslator:
    def __init__(self, target_language: str = "hi"):
        if target_language not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language '{target_language}'. Choose: {list(SUPPORTED_LANGUAGES)}")
        self.target_language = target_language
        self.lang_info = SUPPORTED_LANGUAGES[target_language]
        self.glossary = MEDICAL_GLOSSARY.get(target_language, {})
        self.model = None
        self.tokenizer = None
        self._load_model()

    def _load_model(self):
        try:
            from transformers import MarianMTModel, MarianTokenizer
            import torch
            model_name = self.lang_info["model"]
            logger.info(f"Loading model: {model_name} ...")
            self.tokenizer = MarianTokenizer.from_pretrained(model_name)
            self.model     = MarianMTModel.from_pretrained(model_name)
            if torch.cuda.is_available():
                self.model = self.model.to("cuda")
            self._use_google_fallback = False
            logger.info("Model loaded successfully.")
        except Exception as e:
            logger.warning(f"Model load failed ({e}). Using Google Translate fallback.")
            self._use_google_fallback = True

    def _preprocess(self, text: str) -> str:
        text = re.sub(r"\s+", " ", text).strip()
        # Expanded shortcuts to safeguard translation flow
        abbreviations = {
            r"\bpt\b": "patient",      r"\bdx\b": "diagnosis",
            r"\brx\b": "prescription", r"\bhx\b": "history",
            r"\bbp\b": "blood pressure", r"\bhr\b": "heart rate",
            r"\btemp\b": "temperature",  r"\bwt\b": "weight",
            r"\bht\b": "height",         r"\bbid\b": "twice daily",
            r"\btid\b": "three times daily", r"\bqid\b": "four times daily",
            r"\bprn\b": "as needed",     r"\bstat\b": "immediately",
            r"\bpo\b": "by mouth",       r"\biv\b": "intravenous",
            r"\bim\b": "intramuscular",
            r"\brice\b": "Rest, Ice, Compression, Elevation",
            r"\bmcl\b": "medial collateral ligament sprain"
        }
        for pattern, replacement in abbreviations.items():
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text

    def _detect_medical_terms(self, text: str) -> list:
        text_lower = text.lower()
        return [term for term in self.glossary if term in text_lower]

    def _split_into_segments(self, text: str, max_chars: int = 400) -> list:
        sentences = re.split(r"(?<=[.!?])\s+", text)
        segments, current = [], ""
        for sentence in sentences:
            if len(current) + len(sentence) + 1 <= max_chars:
                current = (current + " " + sentence).strip()
            else:
                if current:
                    segments.append(current)
                current = sentence
        if current:
            segments.append(current)
        return segments or [text]

    def _apply_glossary(self, translated: str, original: str) -> str:
        original_lower = original.lower()
        # Clean up any potential token loops or repetitive characters generated by the model
        translated = re.sub(r'([a-zA-Z0-9])\1{4,}', r'\1', translated)
        for eng_term in sorted(self.glossary, key=len, reverse=True):
            if eng_term in original_lower:
                regional_term = self.glossary[eng_term]
                if regional_term not in translated:
                    # Injects terms smoothly as standard localized parentheses instead of messy trailing arrays
                    translated = f"{translated} ({regional_term})"
        return translated

    def _translate_with_model(self, text: str) -> str:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            ids = self.model.generate(**inputs, num_beams=4, max_length=512, early_stopping=True)
        return self.tokenizer.decode(ids[0], skip_special_tokens=True)

    def _translate_with_google(self, text: str) -> str:
        from googletrans import Translator
        return Translator().translate(text, src="en", dest=self.target_language).text

    def translate(self, text: str) -> TranslationResult:
        start = time.time()
        clean  = self._preprocess(text)
        terms  = self._detect_medical_terms(clean)
        segs   = self._split_into_segments(clean)
        translated_segs = []
        raw_translated_segs = [] # New list for raw translations
        for seg in segs:
            t_raw = self._translate_with_google(seg) if self._use_google_fallback else self._translate_with_model(seg)
            raw_translated_segs.append(t_raw) # Store raw translation
            t_glossary = self._apply_glossary(t_raw, seg)
            translated_segs.append(t_glossary)

        return TranslationResult(
            original_text=text,
            translated_text=" ".join(translated_segs),
            raw_translated_text=" ".join(raw_translated_segs), # Populate new field
            source_language="en",
            target_language=self.target_language,
            model_used="google-translate" if self._use_google_fallback else self.lang_info["model"],
            processing_time_seconds=round(time.time() - start, 2),
            detected_medical_terms=terms,
            confidence_score=round(max(0.0, 1.0 - 0.05 * len(terms)), 2),
            segments=list(zip(segs, translated_segs)),
        )

    def translate_batch(self, texts: list) -> list:
        return [self.translate(t) for t in texts]

    def translate_report(self, report: dict) -> dict:
        translated_report = {}
        for section, original_text in report.items():
            translation_result = self.translate(original_text)
            translated_report[section] = {
                "original": original_text,
                "translated": translation_result.translated_text,
                "raw_translated": translation_result.raw_translated_text,
                "detected_medical_terms": translation_result.detected_medical_terms
            }
        return translated_report

Writing medical_translator.py


In [ ]:
%%writefile evaluation.py
import re
import math
import json # Import json for history persistence
from collections import Counter
from dataclasses import dataclass
from typing import Optional, List, Dict
from medical_translator import MedicalTranscriptionTranslator, MEDICAL_GLOSSARY
from bert_score import score # Import bert_score

# --- Helper functions for history persistence ---
def save_evaluation_history(history: List[Dict], filename: str = "translation_history.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

def load_evaluation_history(filename: str = "translation_history.json") -> List[Dict]:
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def calculate_bert_score(hypothesis: List[str], reference: List[str]) -> List[Dict]:
    # Using a multilingual model that works well for English-Hindi
    # 'microsoft/mdeberta-v3-base' is a good general-purpose multilingual model.
    # We typically report F1 score for BERTScore.
    # The 'score' function returns P, R, F1 tensors. F1 is a tensor of scores for each sentence pair.
    P, R, F1 = score(hypothesis, reference, lang='en', model_type="microsoft/mdeberta-v3-base", verbose=True)
    return [{'p': round(p.item(), 4), 'r': round(r.item(), 4), 'f1': round(f1.item(), 4)} for p, r, f1 in zip(P, R, F1)] # Return a list of dicts with P, R, F1 scores

def medical_term_accuracy(original: str, translation: str, lang: str) -> dict:
    glossary = MEDICAL_GLOSSARY.get(lang, {})
    found_terms_in_original = [term for term in glossary if term in original.lower()]
    correctly_translated_terms = [term for term in found_terms_in_original if glossary[term] in translation]
    missed_terms = [term for term in found_terms_in_original if term not in correctly_translated_terms]

    return {
        "total_medical_terms_in_original": len(found_terms_in_original),
        "correctly_translated_terms_count": len(correctly_translated_terms),
        "accuracy": round(len(correctly_translated_terms)/len(found_terms_in_original), 4) if found_terms_in_original else 1.0,
        "missed_terms": missed_terms,
        "correct_terms": correctly_translated_terms,
    }

@dataclass
class EvalResult:
    sample_id: int
    source_text: str
    hypothesis: str
    reference: Optional[str]
    bert_score_p: Optional[float] # Added BERTScore P
    bert_score_r: Optional[float] # Added BERTScore R
    bert_score_f1: Optional[float] # Added BERTScore F1
    medical_accuracy: dict
    processing_time: float
    target_lang: str

    def to_dict(self):
        return {
            "sample_id": self.sample_id,
            "source_text": self.source_text,
            "hypothesis": self.hypothesis,
            "reference": self.reference,
            "bert_score_p": self.bert_score_p, # Include in dict
            "bert_score_r": self.bert_score_r, # Include in dict
            "bert_score_f1": self.bert_score_f1, # Include in dict
            "medical_accuracy": self.medical_accuracy,
            "processing_time": self.processing_time,
            "target_lang": self.target_lang,
        }

    @classmethod
    def from_dict(cls, data: Dict):
        return cls(
            sample_id=data["sample_id"],
            source_text=data["source_text"],
            hypothesis=data["hypothesis"],
            reference=data["reference"],
            bert_score_p=data.get("bert_score_p"), # Get with .get() for backward compatibility
            bert_score_r=data.get("bert_score_r"), # Get with .get() for backward compatibility
            bert_score_f1=data.get("bert_score_f1"), # Get with .get() for backward compatibility
            medical_accuracy=data["medical_accuracy"],
            processing_time=data["processing_time"],
            target_lang=data["target_lang"],
        )

class TranslationEvaluator:
    def __init__(self, target_language: str = "hi"):
        self.translator      = MedicalTranscriptionTranslator(target_language)
        self.target_language = target_language

    def evaluate_dataset(self, dataset: list) -> List[EvalResult]:
        results = []
        hypotheses = []
        references = []
        source_texts = []

        for item in dataset:
            src = item["source"]
            ref = item.get("reference")
            res = self.translator.translate(src)
            hypotheses.append(res.translated_text)
            references.append(ref)
            source_texts.append(src)

        bert_scores = []
        if all(r is not None for r in references):
            # Calculate BERTScore once for all hypotheses and references
            bert_scores = calculate_bert_score(hypotheses, references)

        for idx, (src, ref, hypothesis) in enumerate(zip(source_texts, references, hypotheses)):
            res = self.translator.translate(src) # Re-translate to get other metrics
            current_bert_score = bert_scores[idx] if bert_scores else {'p': None, 'r': None, 'f1': None}
            results.append(EvalResult(
                sample_id=idx,
                source_text=src,
                hypothesis=hypothesis,
                reference=ref,
                bert_score_p=current_bert_score['p'], # Assign BERTScore P
                bert_score_r=current_bert_score['r'], # Assign BERTScore R
                bert_score_f1=current_bert_score['f1'], # Assign BERTScore F1
                medical_accuracy=medical_term_accuracy(src, hypothesis, self.target_language),
                processing_time=res.processing_time_seconds,
                target_lang=self.target_language,
            ))
        return results

    def aggregate_metrics(self, results: List[EvalResult]) -> dict:
        bert_scores_p = [r.bert_score_p for r in results if r.bert_score_p is not None]
        bert_scores_r = [r.bert_score_r for r in results if r.bert_score_r is not None]
        bert_scores_f1 = [r.bert_score_f1 for r in results if r.bert_score_f1 is not None]
        return {
            "num_samples":                  len(results),
            "avg_bert_score_p":             round(sum(bert_scores_p)/len(bert_scores_p), 4) if bert_scores_p else "N/A", # Add average BERTScore P
            "avg_bert_score_r":             round(sum(bert_scores_r)/len(bert_scores_r), 4) if bert_scores_r else "N/A", # Add average BERTScore R
            "avg_bert_score_f1":            round(sum(bert_scores_f1)/len(bert_scores_f1), 4) if bert_scores_f1 else "N/A", # Add average BERTScore F1
            "avg_medical_term_accuracy":    round(sum(r.medical_accuracy["accuracy"] for r in results)/len(results), 4) if results else "N/A",
            "avg_processing_time_seconds":  round(sum(r.processing_time for r in results)/len(results), 3) if results else "N/A",
        }

    def generate_html_report(self, results: List[EvalResult], output_path: str = "report.html"):
        if not results:
            print("No evaluation results to generate report.")
            return

        m = self.aggregate_metrics(results)
        rows = "".join(f"""
          <tr>
            <td>{r.sample_id}</td>
            <td>{r.source_text[:100]}…</td>
            <td>{r.hypothesis[:100]}…</td>
            <td>{f"{r.bert_score_p:.4f}" if r.bert_score_p is not None else "—"}</td> <!-- Add BERTScore P to table -->
            <td>{f"{r.bert_score_r:.4f}" if r.bert_score_r is not None else "—"}</td> <!-- Add BERTScore R to table -->
            <td>{f"{r.bert_score_f1:.4f}" if r.bert_score_f1 is not None else "—"}</td> <!-- Add BERTScore F1 to table -->
            <td>{r.medical_accuracy['accuracy']:.2%}</td>
            <td>{r.processing_time:.2f}s</td>
          </tr>""" for r in results)

        html = f"""<!DOCTYPE html><html><head><meta charset=\"UTF-8\"><title>Medical Translation Report</title><style>  body{{font-family:sans-serif;background:#f8fafc;padding:2rem;color:#1e293b}}  h1{{border-bottom:3px solid #3b82f6;padding-bottom:.5rem}}  .metrics{{display:flex;gap:1rem;flex-wrap:wrap;margin:1.5rem 0}}  .card{{background:white;border-radius:10px;padding:1rem 2rem;box-shadow:0 2px 8px rgba(0,0,0,.07);text-align:center}}  .value{{font-size:2rem;font-weight:700;color:#3b82f6}}  .label{{font-size:.85rem;color:#64748b}}  table{{width:100%;border-collapse:collapse;background:white;border-radius:10px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.07)}}  th{{background:#1e40af;color:white;padding:.7rem 1rem;text-align:left}}  td{{padding:.6rem 1rem;border-bottom:1px solid #e2e8f0;font-size:.83rem}}  tr:hover td{{background:#eff6ff}}</style></head><body><h1>🏥 Medical Translation — Evaluation Report</h1><div class=\"metrics\">  <div class=\"card\"><div class=\"value\">{m['num_samples']}</div><div class=\"label\">Samples</div></div>  <div class=\"card\"><div class=\"value\">{m['avg_bert_score_p']}</div><div class=\"label\">Avg BERTScore P</div></div>  <div class=\"card\"><div class=\"value\">{m['avg_bert_score_r']}</div><div class=\"label\">Avg BERTScore R</div></div>  <div class=\"card\"><div class=\"value\">{m['avg_bert_score_f1']}</div><div class=\"label\">Avg BERTScore F1</div></div>  <div class=\"card\"><div class=\"value\">{m['avg_medical_term_accuracy']:.2%}</div><div class=\"label\">Med. Term Accuracy</div></div>  <div class=\"card\"><div class=\"value\">{m['avg_processing_time_seconds']}s</div><div class=\"label\">Avg Time</div></div></div><table><thead><tr><th>#</th><th>Source</th><th>Translation</th><th>BERTScore P</th><th>BERTScore R</th><th>BERTScore F1</th><th>Med. Acc.</th><th>Time</th></tr></thead><tbody>{rows}</tbody></table></body></html>"""

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"Report saved → {output_path}")

Writing evaluation.py


In [ ]:
%%writefile demo.py
import argparse
import os # Import os for file path checks
from medical_translator import MedicalTranscriptionTranslator
from evaluation import medical_term_accuracy, TranslationEvaluator, EvalResult, save_evaluation_history, load_evaluation_history, calculate_bert_score # Import calculate_bert_score
import re # for gibberish check

HISTORY_FILE = "translation_history.json"

SAMPLES = [
    """Patient is a 45-year-old male with chest pain and shortness of breath.
    History of hypertension and diabetes. BP 150/95. HR 88.
    Prescription: Metformin 500 mg bid, Amlodipine 5 mg once daily.""",
    """32-year-old female with fever and headache for 5 days.
    Diagnosis: Viral fever. Treatment: Paracetamol 500 mg prn,
    adequate hydration. Return if rash or bleeding develops.""",
    """Knee pain after sports injury. X-ray: no fracture.
    Impression: Grade II MCL sprain. Treatment: RICE,
    Ibuprofen 400 mg tid, physiotherapy referral.""",
]

REPORT = {
    "chief_complaint": "Severe headache with nausea and vomiting for 2 days",
    "history":         "History of migraine and hypertension. No known drug allergy.",
    "diagnosis":       "Hypertensive urgency with migraine",
    "treatment":       "Amlodipine 10 mg once daily. Sumatriptan 50 mg prn for migraine.",
    "follow_up":       "Review in one week. Monitor blood pressure daily.",
}

EVAL_DATA = [
    {"source": "Patient has fever and headache. Diagnosis: viral infection.",
     "reference": "रोगी को बुखार और सिरदर्द है। निदान: वायरल संक्रमण।"},
    {"source": "Blood pressure is high. Prescription: antihypertensive tablet once daily.",
     "reference": "रक्तचाप अधिक है। नुस्खा: उच्चरक्तचापरोधी गोली दिन में एक बार।"},
    {"source": "Patient has diabetes. Take insulin injection as prescribed.",
     "reference": "रोगी को मधुमेह है। निर्देशानुसार इंसुलिन इंजेक्शन लें।"},
]

def contains_hindi(text):
    # Basic check if the text contains Hindi characters
    return bool(re.search(r'[क-ह]', text))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--eval", action="store_true")
    parser.add_argument("--custom_text", type=str, help="Custom English text to translate.")
    parser.add_argument("--reference_text", type=str, help="Optional reference Hindi text for custom translation evaluation.")
    args = parser.parse_args()

    translator = MedicalTranscriptionTranslator(target_language="hi")
    lang_name  = "Hindi"

    if args.custom_text:
        print(f"\n{'═'*65}")
        print(f"  Interactive Medical Translation  |  English → {lang_name}")
        print(f"{'═'*65}")
        print("\nNote on input quality:")
        print("  - For best results, keep the input text concise (e.g., 1-2 sentences).")
        print("  - Focus on medical terminology relevant to the glossary (e.g., 'hypertension', 'fracture').")
        print("  - Avoid overly complex sentence structures or general conversation.")
        print("  - Extremely long texts or non-medical content might lead to less accurate translations or irrelevant glossary applications.")

        print(f"\n[Custom Input]")
        custom_input = args.custom_text
        print(f"  EN : {custom_input}")

        # Perform translation
        translation_result = translator.translate(custom_input)

        # Gibberish check
        # A simple heuristic: if raw translated text is very short OR
        # if it doesn't contain any Hindi characters and is significantly different from original (or too short),
        # it might indicate a translation failure.
        if (len(translation_result.raw_translated_text.strip()) < len(custom_input.strip()) * 0.2 and len(custom_input.strip()) > 10) or \
           (not contains_hindi(translation_result.raw_translated_text) and custom_input.strip().lower() != translation_result.raw_translated_text.strip().lower()):
            print("\n⚠️  WARNING: The translated text appears to be gibberish or very poor quality. Please check your input and consider the guidelines.")
            # We will still proceed to record it for history, but warn the user.

        print(f"  HI (Model w/o Glossary): {translation_result.raw_translated_text}")
        print(f"  HI (Model w/ Glossary) : {translation_result.translated_text}")
        print(f"  Terms detected         : {translation_result.detected_medical_terms}")
        print(f"  Time                   : {translation_result.processing_time_seconds}s")

        medical_acc = medical_term_accuracy(custom_input, translation_result.translated_text, translator.target_language)
        print(f"  Medical Term Accuracy: {medical_acc['accuracy']:.2%} (Missed: {medical_acc['missed_terms']})")

        current_bert_score_p = None
        current_bert_score_r = None
        current_bert_score_f1 = None # Initialize BERTScore
        if args.reference_text:
            # Calculate BERTScore for custom translation
            bert_score_results = calculate_bert_score([translation_result.translated_text], [args.reference_text])
            if bert_score_results:
                current_bert_score_p = bert_score_results[0]['p']
                current_bert_score_r = bert_score_results[0]['r']
                current_bert_score_f1 = bert_score_results[0]['f1'] # Extract the single F1 score
            print(f"  BERTScore P            : {current_bert_score_p:.4f}")
            print(f"  BERTScore R            : {current_bert_score_r:.4f}")
            print(f"  BERTScore F1           : {current_bert_score_f1:.4f}")
            print(f"  Reference Text         : {args.reference_text}")

        # Load existing history
        history_dicts = load_evaluation_history(HISTORY_FILE)
        current_id = len(history_dicts) # Assign new ID for current entry

        # Create an EvalResult for the current translation and convert to dict for saving
        current_eval_result = EvalResult(
            sample_id=current_id,
            source_text=custom_input,
            hypothesis=translation_result.translated_text,
            reference=args.reference_text, # Use provided reference
            bert_score_p=current_bert_score_p, # Store BERTScore P
            bert_score_r=current_bert_score_r, # Store BERTScore R
            bert_score_f1=current_bert_score_f1, # Store BERTScore F1
            medical_accuracy=medical_acc,
            processing_time=translation_result.processing_time_seconds,
            target_lang=translator.target_language,
        )
        history_dicts.append(current_eval_result.to_dict())
        save_evaluation_history(history_dicts, HISTORY_FILE)
        print(f"Translation saved to history: {HISTORY_FILE}")

        # Regenerate report from full history
        all_eval_results = [EvalResult.from_dict(d) for d in history_dicts]
        evaluator = TranslationEvaluator(target_language=translator.target_language) # Re-initialize to ensure it has the translator
        evaluator.generate_html_report(all_eval_results, "report_hi.html")
        print(f"Updated HTML report → report_hi.html with all custom translations.")


        print("\n✅  Done with custom translation.\n")

    else: # Existing logic for SAMPLES, REPORT, batch, eval
        print(f"\n{'═'*65}")
        print(f"  Medical Transcription Translator  |  English → {lang_name}")
        print(f"{'═'*65}")
        for i, text in enumerate(SAMPLES):
            result = translator.translate(text)
            print(f"\n[Sample {i+1}]")
            print(f"  EN : {result.original_text.strip()[:180]}\n") # Added newline for clarity
            print(f"  HI : {result.translated_text}\n") # Added newline for clarity
            print(f"  Terms detected : {result.detected_medical_terms}")
            print(f"  Time : {result.processing_time_seconds}s")

        print(f"\n{'─'*65}\n  Structured Medical Report\n{'─'*65}")
        report_out = translator.translate_report(REPORT)
        for section, data in report_out.items():
            print(f"\n  {section.upper()}")
            print(f"    EN : {data['original']}\n") # Added newline for clarity
            print(f"    HI : {data['translated']}\n") # Added newline for clarity

        print(f"\n{'─'*65}\n  Batch Translation\n{'─'*65}")
        batch = [
            "Take this antibiotic capsule twice daily for seven days.",
            "Patient was discharged after successful surgery.",
            "Allergy to penicillin noted. Dosage of medication adjusted.",
        ]
        for r in translator.translate_batch(batch):
            print(f"\n  EN : {r.original_text}\n") # Added newline for clarity
            print(f"  HI : {r.translated_text}\n") # Added newline for clarity

        if args.eval:
            print(f"\n{'─'*65}\n  Evaluation Metrics\n{'─'*65}")
            evaluator = TranslationEvaluator(target_language="hi")
            results   = evaluator.evaluate_dataset(EVAL_DATA)
            metrics   = evaluator.aggregate_metrics(results)

            print(f"\n  Avg BERTScore P         : {metrics['avg_bert_score_p']}") # Print average BERTScore P
            print(f"  Avg BERTScore R         : {metrics['avg_bert_score_r']}") # Print average BERTScore R
            print(f"  Avg BERTScore F1        : {metrics['avg_bert_score_f1']}") # Print average BERTScore F1
            print(f"  Avg Medical Term Acc.   : {metrics['avg_medical_term_accuracy']:.2%}")
            print(f"  Avg Processing Time     : {metrics['avg_processing_time_seconds']}s")

            for r in results:
                print(f"\n  [{r.sample_id}] BERTScore P={r.bert_score_p} | BERTScore R={r.bert_score_r} | BERTScore F1={r.bert_score_f1} | Med.Acc={r.medical_accuracy['accuracy']:.2%}") # Print individual BERTScore
                if r.medical_accuracy["missed_terms"]:
                    print(f"      ⚠ Missed: {r.medical_accuracy['missed_terms']}")

            evaluator.generate_html_report(results, "report_hi.html")
            print(f"\n  HTML report → report_hi.html")

        print("\n\n✅  Done.\n")

if __name__ == "__main__":
    main()

Writing demo.py


In [ ]:
!python demo.py --custom_text "Patient presents with acute abdominal pain and nausea. CT scan shows appendicitis." --reference_text "रोगी तीव्र पेट दर्द और मतली के साथ प्रस्तुत होता है। सीटी स्कैन में एपेंडिसाइटिस दिखाया गया है।"

2026-06-17 13:53:59,012 [INFO] NumExpr defaulting to 2 threads.
2026-06-17 13:54:13.972227: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 13:54:14.064492: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 13:54:18,085 [WARNING] Warning: Detected no triton, on systems without Triton certain kernels will not work
2026-06-17 13:54:19,446 [INFO] Loading model: Helsinki-NLP/opus-mt-en-hi ...
tokenizer_config.json: 100% 44.0/44.0 [00:00<00:00, 141kB/s]
source.spm: 100% 812k/812k [00:00<00:00, 25.4MB/s]
target.spm: 100% 1.07M/1.07M [00:00<00:00, 135MB/s]
vocab.json: 2.10MB [00:00, 128MB/s]
config.json: 1.39kB [00:00, 5.39MB/s]
/usr/local/lib/python3.12/dist-packages/tr

### Custom Translations with BERTScore for Evaluation

In [ ]:
custom_translations = [
    {
        "english": "Patient suffering from chronic bronchitis. Advised to stop smoking.",
        "hindi_reference": "रोगी पुरानी ब्रोंकाइटिस से पीड़ित है। धूम्रपान बंद करने की सलाह दी गई।"
    },
    {
        "english": "Diagnosis of pneumonia confirmed by chest X-ray. Started on antibiotics.",
        "hindi_reference": "छाती के एक्स-रे से निमोनिया का निदान हुआ। एंटीबायोटिक्स शुरू किए गए।"
    },
    {
        "english": "Post-operative pain managed with strong pain relievers.",
        "hindi_reference": "ऑपरेशन के बाद का दर्द मजबूत दर्द निवारक दवाओं से नियंत्रित किया गया।"
    },
    {
        "english": "Annual health check-up revealed slightly elevated cholesterol levels.",
        "hindi_reference": "वार्षिक स्वास्थ्य जांच में कोलेस्ट्रॉल का स्तर थोड़ा बढ़ा हुआ पाया गया।"
    },
    {
        "english": "Child presented with high fever and rash. Suspected measles.",
        "hindi_reference": "बच्चा तेज बुखार और दाने के साथ प्रस्तुत हुआ। खसरा का संदेह।"
    },
    {
        "english": "Patient is experiencing severe allergic reaction to shellfish. Administer epinephrine.",
        "hindi_reference": "रोगी को शेलफिश से गंभीर एलर्जी की प्रतिक्रिया हो रही है। एपिनेफ्रीन दें।"
    }
]

for i, item in enumerate(custom_translations):
    print(f"\n--- Processing Custom Translation {i+1} ---")
    custom_text = item['english'].replace('"', '\"') # Escape double quotes
    reference_text = item['hindi_reference'].replace('"', '\"') # Escape double quotes
    !python demo.py --custom_text "{custom_text}" --reference_text "{reference_text}"


--- Processing Custom Translation 1 ---
2026-06-17 13:55:38,574 [INFO] NumExpr defaulting to 2 threads.
2026-06-17 13:55:47.669684: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 13:55:47.785461: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 13:55:51,132 [WARNING] Warning: Detected no triton, on systems without Triton certain kernels will not work
2026-06-17 13:55:52,158 [INFO] Loading model: Helsinki-NLP/opus-mt-en-hi ...
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
2026-06-17 13:55:53,739 [INFO] Model l

In [ ]:
print('Re-running custom translations to update history and report with BERT scores...')
custom_translations = [
    {
        "english": "Patient suffering from chronic bronchitis. Advised to stop smoking.",
        "hindi_reference": "रोगी पुरानी ब्रोंकाइटिस से पीड़ित है। धूम्रपान बंद करने की सलाह दी गई।"
    },
    {
        "english": "Diagnosis of pneumonia confirmed by chest X-ray. Started on antibiotics.",
        "hindi_reference": "छाती के एक्स-रे से निमोनia का निदान हुआ। एंटीबायोटिक्स शुरू किए गए।"
    },
    {
        "english": "Post-operative pain managed with strong pain relievers.",
        "hindi_reference": "ऑपरेशन के बाद का दर्द मजबूत दर्द निवारक दवाओं से नियंत्रित किया गया।"
    },
    {
        "english": "Annual health check-up revealed slightly elevated cholesterol levels.",
        "hindi_reference": "वार्षिक स्वास्थ्य जांच में कोलेस्ट्रॉल का स्तर थोड़ा बढ़ा हुआ पाया गया।"
    },
    {
        "english": "Child presented with high fever and rash. Suspected measles.",
        "hindi_reference": "बच्चा तेज बुखार और दाने के साथ प्रस्तुत हुआ। खसरा का संदेह।"
    },
    {
        "english": "Patient is experiencing severe allergic reaction to shellfish. Administer epinephrine.",
        "hindi_reference": "रोगी को शेलफिश से गंभीर एलर्जी की प्रतिक्रिया हो रही है। एपिनेफ्रीन दें।"
    }
]

for i, item in enumerate(custom_translations):
    print(f"\n--- Processing Custom Translation {i+1} ---")
    custom_text = item['english'].replace('"', '\"') # Escape double quotes
    reference_text = item['hindi_reference'].replace('"', '\"') # Escape double quotes
    !python demo.py --custom_text "{custom_text}" --reference_text "{reference_text}"

Re-running custom translations to update history and report with BERT scores...

--- Processing Custom Translation 1 ---
2026-06-17 13:58:00,009 [INFO] NumExpr defaulting to 2 threads.
2026-06-17 13:58:07.209018: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 13:58:07.335190: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 13:58:10,758 [WARNING] Warning: Detected no triton, on systems without Triton certain kernels will not work
2026-06-17 13:58:11,762 [INFO] Loading model: Helsinki-NLP/opus-mt-en-hi ...
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.war

In [ ]:
print('\n--- Running evaluation for hardcoded samples to ensure report update ---')
!python demo.py --eval


--- Running evaluation for hardcoded samples to ensure report update ---
2026-06-17 14:01:03,351 [INFO] NumExpr defaulting to 2 threads.
2026-06-17 14:01:11.284282: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 14:01:11.366447: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 14:01:14,857 [WARNING] Warning: Detected no triton, on systems without Triton certain kernels will not work
2026-06-17 14:01:15,897 [INFO] Loading model: Helsinki-NLP/opus-mt-en-hi ...
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
2026-

In [ ]:
print('\n--- Verifying content of report_hi.html ---')

with open('report_hi.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Check for BERTScore F1 in the HTML content, looking for numerical values
import re
bert_score_pattern = re.compile(r'Avg BERTScore F1</div>\s*<div class="value">([0-9.]+)')
match = bert_score_pattern.search(html_content)

if match:
    bert_score_value = match.group(1)
    print(f"Found Avg BERTScore F1 in report_hi.html: {bert_score_value}")
    print("It appears the report_hi.html is now correctly displaying numerical BERT scores.")
else:
    print("Could not find numerical BERTScore F1 in report_hi.html. Further investigation may be needed.")

# Also check for individual BERTScore F1 values in the table
individual_bert_score_pattern = re.compile(r'<td>([0-9]+\.[0-9]{4})</td>\s*<!-- Add BERTScore F1 to table -->')
individual_matches = individual_bert_score_pattern.findall(html_content)

if individual_matches:
    print(f"Found individual BERTScore F1 values in the table: {individual_matches[:5]}...")
else:
    print("Could not find individual BERTScore F1 values in the report table.")



--- Verifying content of report_hi.html ---
Could not find numerical BERTScore F1 in report_hi.html. Further investigation may be needed.
Found individual BERTScore F1 values in the table: ['0.8155', '0.6476', '0.7225']...


### Evaluating Hardcoded Samples

In [ ]:
print("\n--- Running evaluation for hardcoded samples ---")
!python demo.py --eval


--- Running evaluation for hardcoded samples ---
2026-06-17 14:02:00,210 [INFO] NumExpr defaulting to 2 threads.
2026-06-17 14:02:07.076547: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 14:02:07.162476: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 14:02:11,454 [WARNING] Warning: Detected no triton, on systems without Triton certain kernels will not work
2026-06-17 14:02:12,597 [INFO] Loading model: Helsinki-NLP/opus-mt-en-hi ...
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
2026-06-17 14:02:14,032 [INFO

In [ ]:
def _apply_glossary(self, translated: str, original: str) -> str:
        original_lower = original.lower()
        # Sort key by length to match multi-word phrases first (e.g., 'blood pressure' before 'blood')
        for eng_term in sorted(self.glossary, key=len, reverse=True):
            if eng_term in original_lower:
                regional_term = self.glossary[eng_term]
                # If the translation missing the key term completely, patch it elegantly inline
                if regional_term not in translated:
                    # Optional: substitute instead of messy trailing arrays
                    translated = f"{translated} ({regional_term})"
        return translated

In [ ]:
from medical_translator import MedicalTranscriptionTranslator

# Instantiate the translator (this will load the model or fallback to Google Translate)
translator = MedicalTranscriptionTranslator(target_language='hi')

# Sample data to demonstrate _apply_glossary
original_text = "Patient has hypertension and diabetes."
translated_text = "रोगी को उच्च रक्तचाप है और मधुमेह है।"

# Call the _apply_glossary method directly
# Note: In a real scenario, this method is called internally by the translate method.
# We are calling it here for demonstration purposes.
result = translator._apply_glossary(translated_text, original_text)
print(f"Original: {original_text}")
print(f"Translated (before glossary): {translated_text}")
print(f"Translated (after glossary): {result}")

original_text_2 = "Knee pain, no fracture. MCL sprain."
translated_text_2 = "घुटने का दर्द, कोई फ्रैक्चर नहीं। एमसीएल मोच।"
result_2 = translator._apply_glossary(translated_text_2, original_text_2)
print(f"\nOriginal: {original_text_2}")
print(f"Translated (before glossary): {translated_text_2}")
print(f"Translated (after glossary): {result_2}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Original: Patient has hypertension and diabetes.
Translated (before glossary): रोगी को उच्च रक्तचाप है और मधुमेह है।
Translated (after glossary): रोगी को उच्च रक्तचाप है और मधुमेह है।

Original: Knee pain, no fracture. MCL sprain.
Translated (before glossary): घुटने का दर्द, कोई फ्रैक्चर नहीं। एमसीएल मोच।
Translated (after glossary): घुटने का दर्द, कोई फ्रैक्चर नहीं। एमसीएल मोच। (हड्डी टूटना)


## Alternative Metric: BERTScore

Given the challenges with BLEU scores being consistently low due to its strict n-gram matching and the specific way glossary terms are appended, a more semantically aware metric like **BERTScore** can provide a better evaluation.

### What is BERTScore?

BERTScore leverages contextual embeddings from pre-trained BERT models to calculate the similarity between tokens in the candidate (hypothesis) and reference sentences. Instead of counting exact word matches, it computes a soft score based on how similar the *meaning* of the words (and their context) are. This means:

*   **Semantic Similarity:** It can identify that 'patient' and 'व्यक्ति' (person) are similar in context, even if they are not the same word, which BLEU would penalize.
*   **Less Sensitive to Word Order:** While order matters, minor rephrasing that maintains meaning is less harshly penalized.
*   **Robust to Synonyms and Paraphrases:** If your model uses a synonym that conveys the same meaning as the reference, BERTScore is more likely to give it credit.
*   **Better Correlation with Human Judgments:** Studies often show that BERTScore correlates better with human evaluations of translation quality compared to BLEU.

### How it addresses the current problem:

With glossary terms being appended, the translated sentences often don't exactly match a human-written reference's structure, leading to very low BLEU scores. BERTScore, by focusing on meaning, can give a higher score if the core medical information and overall message are correctly conveyed, even if the glossary additions make the sentence structure diverge from the reference.

We will integrate BERTScore into our evaluation process to get a more nuanced understanding of translation quality.

In [ ]:
!pip install bert_score

/content
On branch main
nothing to commit, working tree clean
fatal: unable to access 'https://://github.comPranathitejasvi/Automatic-Machine-translation-of-Medical-transcription.git/': URL using bad/illegal format or missing URL
